# 08 Strata: urbanity, vulnerability, tribal status and hazards
The tract-level context the challenge asks about: how urban or rural a tract is, how socially and climatically vulnerable, whether it is tribal, and which hazards it faces. In this project these variables are **audit-only**: they define the groups whose coverage is compared in notebook 09 and later, and they never enter an expectation of how many features a tract should have.

This notebook reads the national strata table, checks coverage and missingness, compares alternative indicators of the same concept (they do not always agree), defines the audit groups and saves `eda_outputs/audit_strata.parquet`.

In [1]:
from common import *
st = pd.read_parquet(tracts_parquet("strata-tract-table"))
ids = pd.concat([pd.read_csv(REF / r / f"{r}-sample-submission.csv", dtype={"GEOID": str}).assign(region=r) for r in REG])[["GEOID", "region"]]
st["GEOID"] = st.GEOID.astype(str)
a = ids.merge(st, on="GEOID", how="left")
print(a.shape, "| scored tracts missing from strata:", int(a.state_usps.isna().sum()))
cov = [c for c in a.columns if c.endswith("_covered")]
print("share of tracts covered by each source:"); print(a.groupby("region")[cov].mean().loc[REG].round(2).T.to_string())

(9794, 233) | scored tracts missing from strata: 0
share of tracts covered by each source:
region              maricopa-az  northern-ca  eastern-ok  south-central-tx  eastern-wa
svi_covered               1.000        1.000       1.000             1.000       1.000
cvi_covered               1.000        1.000       1.000             1.000       1.000
ruca_covered              1.000        1.000       1.000             1.000       1.000
rucc_covered              1.000        1.000       1.000             1.000       1.000
nchs_covered              1.000        1.000       1.000             1.000       1.000
usdm_covered              1.000        1.000       1.000             1.000       1.000
usfs_covered              1.000        1.000       1.000             1.000       1.000
epht_covered              1.000        1.000       1.000             1.000       1.000
carbonplan_covered        1.000        1.000       1.000             1.000       1.000
usgs_covered              1.000        

**Takeaway:** All 9,794 scored tracts are present in the national strata table, and every source reports full coverage for every region. The one exception is the urban-heat-island city layer (UHE), which exists only for tracts inside the cities it lists (38% to 69% of tracts by region) and is therefore a partial layer, not a missing one.

## 1. Urban and rural: four definitions
Census urban/rural share (`ur_class`, `pct_urban`), USDA RUCA, USDA RUCC (metro or not) and the NCHS six-level scheme. They measure different things (population share, commuting, county size), so compare them before picking one.

In [2]:
a["ru_census"] = a.ur_class.map({"Rural": 1, "Urban": 0})
a["ru_ruca"] = (a.ruca_primary >= 4).where(a.ruca_primary.notna() & (a.ruca_primary != 99)).astype(float)
a["ru_rucc"] = (~a.rucc_metro.astype(bool)).astype(float)
a["ru_nchs"] = a.nchs_2013_label.isin(["Micropolitan", "Noncore"]).astype(float)
a["ru_pct50"] = (a.pct_urban < .5).where(a.pct_urban.notna()).astype(float)
R = ["ru_census", "ru_ruca", "ru_rucc", "ru_nchs", "ru_pct50"]
sc = a[a.region.isin(SCORED)]
print("share of scored tracts flagged rural:"); print(sc[R].mean().round(3).to_string())
print("\nby region:"); print(sc.groupby("region")[R].mean().loc[SCORED].round(3).to_string())
print("\nCohen kappa between definitions (scored four):")
def kappa(x, y):
    m = x.notna() & y.notna(); x, y = x[m], y[m]; po = (x == y).mean(); pe = x.mean() * y.mean() + (1 - x.mean()) * (1 - y.mean()); return (po - pe) / (1 - pe)
print(pd.DataFrame({i: {j: kappa(sc[i], sc[j]) for j in R} for i in R}).round(2).to_string())
print("\nmissing: ur_class", int(a.ur_class.isna().sum()), "| pct_urban", int(a.pct_urban.isna().sum()), "| ruca code 99 (not coded):", int((a.ruca_primary == 99).sum()))
print("zero-population tracts without a census class:", int((a.ur_class.isna() & (a.pop_total == 0)).sum()))

share of scored tracts flagged rural:
ru_census   0.183
ru_ruca     0.166
ru_rucc     0.132
ru_nchs     0.134
ru_pct50    0.183

by region:
                  ru_census  ru_ruca  ru_rucc  ru_nchs  ru_pct50
region                                                          
maricopa-az           0.103    0.109    0.051    0.051     0.103
northern-ca           0.249    0.264    0.200    0.200     0.249
eastern-ok            0.345    0.360    0.328    0.332     0.345
south-central-tx      0.165    0.133    0.108    0.110     0.165

Cohen kappa between definitions (scored four):
           ru_census  ru_ruca  ru_rucc  ru_nchs  ru_pct50
ru_census      1.000    0.470    0.460    0.460     1.000
ru_ruca        0.470    1.000    0.780    0.770     0.470
ru_rucc        0.460    0.780    1.000    0.960     0.460
ru_nchs        0.460    0.770    0.960    1.000     0.460
ru_pct50       1.000    0.470    0.460    0.460     1.000

missing: ur_class 27 | pct_urban 27 | ruca code 99 (not coded): 2
zero-po

## 2. Tribal status
Tracts overlapping tribal areas, split into legal (reservations, trust land) and statistical (OTSAs and similar) areas, and how much of the tract the area covers.

In [3]:
tr = sc.groupby("region").agg(tracts=("GEOID", "size"), tribal_any=("tribal_any", "sum"), legal=("tribal_legal", "sum"), statistical=("tribal_statistical", "sum"))
print(tr.loc[SCORED].to_string()); print("\ntribal tracts, scored four:", int(sc.tribal_any.sum()), "| Eastern WA (not scored):", int(a[a.region == "eastern-wa"].tribal_any.sum()))
tt = sc[sc.tribal_any]
print("\nshare of tribal tracts in Eastern OK:", round((tt.region == "eastern-ok").mean(), 3))
print("tribal_pct (fraction of the tract inside a tribal area): median", round(tt.tribal_pct.median(), 3), "| share of tribal tracts with >= 50% overlap:", round((tt.tribal_pct >= .5).mean(), 3), "| min", round(tt.tribal_pct.min(), 3))
print("tribal tracts that are Census-urban:", int((tt.ur_class == "Urban").sum()), "of", len(tt))
print("\nlargest areas by tracts:"); print(tt.aiannh_name.value_counts().head(8).to_string())

                  tracts  tribal_any  legal  statistical
region                                                  
maricopa-az         1593          37     37            0
northern-ca          591          15     13            2
eastern-ok          1192         805     13          792
south-central-tx    6003           1      1            0

tribal tracts, scored four: 858 | Eastern WA (not scored): 16

share of tribal tracts in Eastern OK: 0.938
tribal_pct (fraction of the tract inside a tribal area): median 1.0 | share of tribal tracts with >= 50% overlap: 0.96 | min 0.01
tribal tracts that are Census-urban: 448 of 858

largest areas by tracts:
aiannh_name
Creek OTSA                                         256
Cherokee OTSA                                      152
Chickasaw OTSA                                      93
Choctaw OTSA                                        67
Kiowa-Comanche-Apache-Fort Sill Apache OTSA         63
Cheyenne and Arapaho OTSA                           57
Citi

**Takeaway:** The tribal group is effectively Eastern Oklahoma: 858 scored tracts overlap a tribal area, 805 of them (93.8%) in Eastern OK, against 37 in Maricopa, 15 in Northern CA and 1 in Texas. In Eastern OK 792 of the 805 are statistical areas (OTSAs) and only 13 are legal areas; 96% of tribal tracts lie at least half inside the tribal area. Of the 858, 448 (52%) are Census-urban (the Creek OTSA includes Tulsa), so "tribal" and "rural" are different things. A tribal versus non-tribal comparison is mostly one OTSA region against its surroundings.

## 3. Social and climate vulnerability
CDC Social Vulnerability Index (overall percentile and four themes) and the Climate Vulnerability Index. Missingness, scale, and how the two relate.

In [4]:
V = ["svi_overall", "svi_socioeconomic", "svi_household", "svi_minority", "svi_housing_transport", "cvi_overall", "cvi_baseline", "cvi_climate"]
print(sc[V].describe().T[["count", "mean", "min", "50%", "max"]].round(3).to_string())
print("\nSVI missing (scored):", int(sc.svi_overall.isna().sum()), "| of which zero-population:", int((sc.svi_overall.isna() & (sc.pop_total == 0)).sum()), "| by region:", sc[sc.svi_overall.isna()].region.value_counts().to_dict())
print("\nSpearman:"); print(sc[V].corr(method="spearman").round(2).to_string())
print("\nmedian by region:"); print(sc.groupby("region")[["svi_overall", "cvi_overall"]].median().loc[SCORED].round(3).to_string())

                          count  mean   min   50%   max
svi_overall           9,294.000 0.568 0.000 0.612 1.000
svi_socioeconomic     9,294.000 0.577 0.000 0.615 1.000
svi_household         9,295.000 0.560 0.000 0.599 1.000
svi_minority          9,331.000 0.635 0.000 0.648 0.995
svi_housing_transport 9,295.000 0.492 0.000 0.491 1.000
cvi_overall           9,377.000 0.519 0.326 0.522 0.736
cvi_baseline          9,377.000 0.520 0.231 0.522 0.808
cvi_climate           9,377.000 0.518 0.330 0.513 0.774

SVI missing (scored): 85 | of which zero-population: 26 | by region: {'south-central-tx': 45, 'maricopa-az': 29, 'eastern-ok': 11}

Spearman:
                       svi_overall  svi_socioeconomic  svi_household  svi_minority  svi_housing_transport  cvi_overall  cvi_baseline  cvi_climate
svi_overall                  1.000              0.910          0.740         0.610                  0.760        0.740         0.810        0.110
svi_socioeconomic            0.910              1.000        

**Takeaway:** SVI is missing for 85 scored tracts (Texas 45, Maricopa 29, Eastern OK 11), 26 of them with no residents, which leaves 9,294 with an SVI; CVI is available for 9,377. Overall SVI and CVI correlate at 0.74, but that comes through CVI's baseline pillar (0.81 with SVI); the climate pillar is nearly unrelated to SVI (0.11). "High social vulnerability" and "high climate vulnerability" are therefore different groups. Median SVI rises from 0.52 in Maricopa to 0.64 in Texas.

## 4. Hazards: do alternative indicators agree?
Heat, drought and wildfire each have several indicators from different sources. Spearman correlation within each family, over the scored four regions pooled and within region. Low agreement means a finding about "high heat tracts" depends on which indicator defines them.

In [5]:
H = {"heat": ["epht_heat_days_summer", "gehe_wbgt28_days_mean", "hwd_maxtemp_mean", "hwd_heatindex_mean"],
     "drought": ["usdm_summer_dsci", "pmdi_dry", "spi12_dry", "spi03_pct_months_d2"],
     "wildfire": ["usfs_WHP_mean", "usfs_RPS_mean", "carbonplan_rps_2011_mean", "mtbs_wildfire_burned_pct_land", "nifc_wildfire_burned_pct_land", "fod_ignition_density"]}
sc = sc.assign(pmdi_dry=-sc.pmdi_mean, spi12_dry=-sc.spi12_mean)  # flip so that higher = drier for every drought indicator
for k, cols in H.items():
    print(f"== {k}: pooled Spearman"); print(sc[cols].corr(method="spearman").round(2).to_string())
    wr = pd.DataFrame({r: sc[sc.region == r][cols].corr(method="spearman").where(np.triu(np.ones((len(cols),) * 2), 1).astype(bool)).stack().mean() for r in SCORED}, index=["mean pairwise"]).round(2)
    print("within-region mean pairwise:", wr.iloc[0].to_dict(), "\n")

== heat: pooled Spearman
                       epht_heat_days_summer  gehe_wbgt28_days_mean  hwd_maxtemp_mean  hwd_heatindex_mean
epht_heat_days_summer                  1.000                  0.670            -0.170              -0.490
gehe_wbgt28_days_mean                  0.670                  1.000            -0.330              -0.370
hwd_maxtemp_mean                      -0.170                 -0.330             1.000              -0.210
hwd_heatindex_mean                    -0.490                 -0.370            -0.210               1.000
within-region mean pairwise: {'maricopa-az': -0.26, 'northern-ca': 0.01, 'eastern-ok': 0.17, 'south-central-tx': -0.1} 

== drought: pooled Spearman
                     usdm_summer_dsci  pmdi_dry  spi12_dry  spi03_pct_months_d2
usdm_summer_dsci                1.000     0.760      0.810                0.780
pmdi_dry                        0.760     1.000      0.900                0.650
spi12_dry                       0.810     0.900      1.0

**Takeaway:** Alternative indicators of the same hazard do not agree well. Wildfire: the USFS and CarbonPlan risk scores agree (0.74 to 0.95), while burned area (0.37 to 0.57) and ignition density (0.03 to 0.42) agree less. Drought: with signs aligned (higher = drier) the four indicators agree strongly across all tracts (0.65 to 0.90), but within a single region the average agreement falls to -0.04 (Maricopa), 0.21 (Eastern OK), 0.27 (Northern CA) and 0.58 (Texas), so most of that agreement is the dry-west against wet-east gradient. Heat: the CDC EPHT summer heat days and the NASA WBGT days agree (0.67), but the heat-wave day metrics correlate negatively with them (-0.49 to -0.17), and the within-region averages are -0.26 to 0.17. Findings about "high heat" or "high drought" tracts will depend on which indicator defines them.

## 5. How each hazard differs by region and urbanity
Medians of the primary indicator per region and urban class, plus coverage of the urban-heat-island city layer (UHE), which exists only for tracts inside the listed cities.

In [6]:
P = ["epht_heat_days_summer", "gehe_wbgt28_days_mean", "usdm_summer_dsci", "pmdi_mean", "usfs_WHP_mean", "usfs_RPS_mean"]
print(sc.groupby("region")[P].median().loc[SCORED].round(2).to_string()); print()
print(sc.assign(ur=sc.ur_class.fillna("unclassified")).groupby("ur")[P].median().round(2).to_string())
print("\nUHE city layer: share of tracts inside a listed city:", sc.groupby("region").uhe_covered.mean().loc[SCORED].round(2).to_dict())
print("hazard-rich tracts (not NaN) in primary indicators:", {c: int(sc[c].notna().sum()) for c in P})

                  epht_heat_days_summer  gehe_wbgt28_days_mean  usdm_summer_dsci  pmdi_mean  usfs_WHP_mean  usfs_RPS_mean
region                                                                                                                   
maricopa-az                      15.110                 82.630           175.590     -1.160         66.060          0.000
northern-ca                      11.000                  7.310           216.030     -1.440        108.010          0.020
eastern-ok                       11.330                 33.780            63.330      1.130         12.630          0.020
south-central-tx                 15.220                 59.590            85.210      0.850         18.840          0.000

              epht_heat_days_summer  gehe_wbgt28_days_mean  usdm_summer_dsci  pmdi_mean  usfs_WHP_mean  usfs_RPS_mean
ur                                                                                                                   
Rural                        13

**Takeaway:** Regions differ more than tract types do. Median WBGT 28 days per year are 82.6 in Maricopa against 7.3 in Northern CA; the Drought Monitor index is 176 in Maricopa and 216 in Northern CA against 63 in Eastern OK; wildfire hazard potential is 108 in Northern CA and 66 in Maricopa against 13 in Eastern OK. The EPHT summer heat days barely differ between regions (11.0 to 15.2), another sign that indicators measure different things. By urbanity, wildfire hazard is five times higher in rural tracts (92 against 18), while the Drought Monitor index is about the same (98 and 94) and WBGT days are higher in urban tracts (55 against 40).

## 6. Audit groups
One primary indicator per concept, with the alternatives kept as sensitivity checks. "High" means the top quartile among the scored four regions pooled (so groups are comparable across regions); the within-region top quartile is saved too, because a tract can be high-heat for Northern California without being high-heat for Maricopa.

| Group | Primary definition | Sensitivity |
|---|---|---|
| rural | Census `ur_class == Rural` (under 50% of residents urban) | RUCA >= 4, RUCC non-metro, NCHS micro/noncore |
| tribal | any overlap with a tribal area | legal-only, statistical-only |
| high SVI | `svi_overall` top quartile | CVI overall top quartile |
| high heat | `epht_heat_days_summer` top quartile | WBGT 28 days, max-temperature heat-wave days |
| high drought | `usdm_summer_dsci` top quartile | PMDI (low), SPI12 (low) |
| high wildfire | `usfs_WHP_mean` top quartile | RPS, burned area |

In [7]:
def hi(df, col, sign=1, within=False):
    x = sign * df[col]
    if within: return x.groupby(df.region).transform(lambda v: v >= v.quantile(.75)).where(x.notna())
    return (x >= x[df.region.isin(SCORED)].quantile(.75)).where(x.notna())
A = a[["GEOID", "region", "state_usps", "pop_total", "pct_urban", "ur_class"]].copy()
A["rural"] = a.ru_census == 1
for k in ["ru_ruca", "ru_rucc", "ru_nchs"]: A[f"rural_{k[3:]}"] = a[k]
A["tribal"] = a.tribal_any; A["tribal_legal"] = a.tribal_legal; A["tribal_statistical"] = a.tribal_statistical
spec = {"svi": ("svi_overall", 1), "cvi": ("cvi_overall", 1), "heat": ("epht_heat_days_summer", 1), "heat_wbgt": ("gehe_wbgt28_days_mean", 1), "heat_tmax": ("hwd_maxtemp_mean", 1),
        "drought": ("usdm_summer_dsci", 1), "drought_pmdi": ("pmdi_mean", -1), "drought_spi": ("spi12_mean", -1),
        "fire": ("usfs_WHP_mean", 1), "fire_rps": ("usfs_RPS_mean", 1), "fire_burned": ("mtbs_wildfire_burned_pct_land", 1)}
for k, (col, sg) in spec.items():
    A[col] = a[col]; A[f"high_{k}"] = hi(a, col, sg); A[f"high_{k}_wr"] = hi(a, col, sg, within=True)
A.to_parquet(OUT / "audit_strata.parquet", index=False)
G = ["rural", "tribal", "high_svi", "high_heat", "high_drought", "high_fire"]
sz = A[A.region.isin(SCORED)].groupby("region")[G].agg(lambda x: int(x.fillna(False).astype(bool).sum())).loc[SCORED]; sz["tracts"] = A[A.region.isin(SCORED)].region.value_counts()
print(sz.to_string()); print("\ntotals (scored four):", sz.sum().to_dict()); print("saved", A.shape)
print("within-region top-quartile versions, tracts per region:"); print(A[A.region.isin(SCORED)].groupby("region")[[g + "_wr" for g in G[2:]]].agg(lambda x: int(x.fillna(False).astype(bool).sum())).loc[SCORED].to_string())

                  rural  tribal  high_svi  high_heat  high_drought  high_fire  tracts
region                                                                               
maricopa-az         163      37       327         62          1505        780    1593
northern-ca         147      15       116          0           588        362     591
eastern-ok          411     805       256          0            66        267    1192
south-central-tx    991       1      1626       2533           186        936    6003

totals (scored four): {'rural': 1712, 'tribal': 858, 'high_svi': 2325, 'high_heat': 2595, 'high_drought': 2345, 'high_fire': 2345, 'tracts': 9379}
saved (9794, 46)
within-region top-quartile versions, tracts per region:
                  high_svi_wr  high_heat_wr  high_drought_wr  high_fire_wr
region                                                                    
maricopa-az               391           492              399           399
northern-ca               148         

**Takeaway:** Pooled top-quartile thresholds make the groups region-specific. Of the 2,595 "high heat" tracts 2,533 (97.6%) are in Texas and none are in Northern CA or Eastern OK; 64% of "high drought" tracts are in Maricopa, 70% of "high SVI" tracts in Texas, and 94% of tribal tracts in Eastern OK. Group sizes are rural 1,712, tribal 858, high SVI 2,325, high heat 2,595, high drought 2,345 and high wildfire 2,345. The within-region top-quartile versions, saved alongside, have 148 to 1,517 tracts per region and are the fairer basis for comparing groups inside a region; the pooled versions suit description across regions.

## 7. Overlap of the groups
The challenge's hypothesis is about overlap (rural and tribal and drought-prone, and so on). How many tracts sit in more than one group, and do the groups mostly coincide with a single region?

In [8]:
F = A[A.region.isin(SCORED)][G].fillna(False).astype(bool)
print("tract counts in the 2-way overlaps:"); print(pd.DataFrame({i: {j: int((F[i] & F[j]).sum()) for j in G} for i in G}).to_string())
print("\nnumber of groups a tract belongs to:", F.sum(axis=1).value_counts().sort_index().to_dict())
print("\nregion makeup of each group (%):"); print((100 * pd.DataFrame({g: A[A.region.isin(SCORED)][F[g].values].region.value_counts(normalize=True) for g in G}).fillna(0)).round(1).loc[SCORED].to_string())
print("\ntracts in 3+ groups by region:", A[A.region.isin(SCORED)][F.sum(axis=1).values >= 3].region.value_counts().to_dict())

tract counts in the 2-way overlaps:
              rural  tribal  high_svi  high_heat  high_drought  high_fire
rural          1712     410       245        281           404       1065
tribal          410     858       213          3            93        287
high_svi        245     213      2325        771           488        374
high_heat       281       3       771       2595           154        214
high_drought    404      93       488        154          2345       1119
high_fire      1065     287       374        214          1119       2345

number of groups a tract belongs to: {0: 2048, 1: 3577, 2: 2813, 3: 807, 4: 117, 5: 14, 6: 3}

region makeup of each group (%):


                  rural  tribal  high_svi  high_heat  high_drought  high_fire
region                                                                       
maricopa-az       9.500   4.300    14.100      2.400        64.200     33.300
northern-ca       8.600   1.700     5.000      0.000        25.100     15.400
eastern-ok       24.000  93.800    11.000      0.000         2.800     11.400
south-central-tx 57.900   0.100    69.900     97.600         7.900     39.900

tracts in 3+ groups by region: {'eastern-ok': 274, 'south-central-tx': 243, 'maricopa-az': 242, 'northern-ca': 182}


**Takeaway:** The groups overlap heavily in some pairs and hardly at all in others. Of the 1,712 rural tracts, 1,065 (62%) are also high wildfire; drought and wildfire share 1,119 tracts; tribal and high heat share only 3. Across the scored four, 2,048 tracts (22%) are in no group, 3,577 in one, 2,813 in two and 941 in three or more, spread across all regions (Eastern OK 274, Texas 243, Maricopa 242, Northern CA 182). Compound cases the challenge cares about (for example rural, tribal and drought-prone) are therefore small groups, which limits how precisely any gap can be estimated for them.